In [0]:
from pyspark.sql import functions as F

def construir_dim_tiempo(fecha_min: str, fecha_max: str):
    """
    Genera una fila por día en el rango, con año, mes, trimestre y clave sustituta.
    """
    # 1. Generamos la secuencia de fechas continuas usando Spark SQL
    df_fechas = spark.sql(f"SELECT explode(sequence(to_date('{fecha_min}'), to_date('{fecha_max}'), interval 1 day)) AS fecha")
    
    # 2. Derivamos las columnas de la dimensión
    dim_tiempo = df_fechas.select(
        F.date_format("fecha", "yyyyMMdd").cast("int").alias("id_tiempo"),
        "fecha",
        F.year("fecha").alias("anio"),
        F.month("fecha").alias("mes"),
        F.quarter("fecha").alias("trimestre"),
        F.date_format("fecha", "MMMM").alias("nombre_mes") # Enero, Febrero, etc. (En el idioma del clúster)
    )
    
    return dim_tiempo

In [0]:
# Como nuestro MVP es el 2005, creamos la dimensión para ese año. 
df_dim_tiempo = construir_dim_tiempo("2005-01-01", "2005-12-31")

# Guardamos en la capa Gold
# Primero creamos la base de datos si no existe
spark.sql("CREATE DATABASE IF NOT EXISTS gold;")

df_dim_tiempo.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.dim_tiempo")

print(f"Filas generadas en Dim Tiempo: {df_dim_tiempo.count()}")
display(df_dim_tiempo.limit(5))

Filas generadas en Dim Tiempo: 365


id_tiempo,fecha,anio,mes,trimestre,nombre_mes
20050101,2005-01-01,2005,1,1,January
20050102,2005-01-02,2005,1,1,January
20050103,2005-01-03,2005,1,1,January
20050104,2005-01-04,2005,1,1,January
20050105,2005-01-05,2005,1,1,January
